# 31 · Интеграция Фаз C/D/E (риск-портфель yглубление)

C1: стратифицированные метрики sensor по типам датчиков (пер-типовая разница).
C2: эхо-верификация fire (разброс времени между дымовыми каналами подтверждённых серий).
C3: последовательность access «дверь -> движение» (период в минутах).
C4: value-пайплайн — статус (температура для fire отрицательный, §12-15).
C5: L2 объектный индекс риска (агрегация над holdout + реестр каналов).
D: отчёт соответствия ТЗ (report_tz_compliance).
E: статус адаптеров внешних данных.

In [1]:
%matplotlib inline
import json
import pathlib
import sys

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import inference_contract as ic
import data_utils as du
from tte_experiments import pr_at_horizon

DATASET = du.find_dataset_dir()
MODELS = RESEARCH / "models"


In [2]:
# C1. Стратифицированные метрики sensor по типам датчиков
df = pd.read_parquet(MODELS / "tte_sensor_holdout.parquet")
rows = []
for t, g in df.groupby("тип_датчика"):
    y24 = ((g["event_flag"] == 1) & (g["obs_days"] <= 1.01)).to_numpy(int)
    p24 = g["p24"].to_numpy(float)
    y30 = ((g["event_flag"] == 1) & (g["obs_days"] <= 30.01)).to_numpy(int)
    pr24 = pr_at_horizon(y24, p24)
    pr30 = pr_at_horizon(y30, g["risk30"].to_numpy(float))
    rows.append({
        "тип": t, "n": len(g),
        "база24%": round(pr24["base"] * 100, 3),
        "PR-AUC24": round(pr24["pr_auc"], 3),
        "база30%": round(pr30["base"] * 100, 1),
        "PR-AUC30": round(pr30["pr_auc"], 3),
        "мед.exp_days": round(g["exp_days"].median(), 1),
    })
c1 = pd.DataFrame(rows).sort_values("n", ascending=False)
print(c1.to_string(index=False))


D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib

D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Downloads_D\lct_a8\research\.venv\Lib

               тип    n  база24%  PR-AUC24  база30%  PR-AUC30  мед.exp_days
    Газовый датчик 7064    0.283     0.004      6.7     0.106          29.1
       Датчик дыма 4299    0.861     0.018      6.7     0.142          28.8
Датчик температуры 2623    0.038     0.011      2.2     0.140          29.8
          КД Дверь 2616    0.076     0.001      1.0     0.062          29.8
   Датчик движения 2566    0.039     0.000      0.3     0.010          29.8
             КД АВ  476    0.000     0.000      2.3     0.046          29.1
   Тепловой датчик  273    0.000     0.000      0.7     0.086          29.8
            Стекло   40    0.000     0.000      0.0     0.000          29.0
            КД Люк   30    0.000     0.000      0.0     0.000          29.7
 Датчик затопления   12    0.000     0.000      0.0     0.000          29.7


In [3]:
# C2/C3. Эхо fire и последовательность access (из журнала, dataset/*.json)
fp = DATASET / "echo_sequential_stats.json"
if fp.exists():
    st = json.loads(fp.read_text(encoding="utf-8"))
    print(chr(10).join(str(k) + ": " + str(v) for k, v in st.items()))
else:
    print("echo_sequential_stats.json ещё не готов — ждём фоновый расчёт")


fire подтверждённых 6ч-бакетов (>=2 канала): 897
разброс <=5 мин, %: 19.29
разброс 5-30 мин, %: 24.19
разброс >30 мин, %: 56.52
медиана разброса, мин: 40.5
дверных событий: 1458473
дверь->движение <=30 мин, шт: 1171711
доля дверь->движение <=1 мин, %: 75.14
доля дверь->движение <=5 мин, %: 88.59
медиана дверь->движение, мин: 0.08


In [4]:
# C5. L2 объектный индекс риска (holdout + реестр каналов -> ид_объект)
ref = du.load_ref_channels()[["ид_канала_данных", "ид_объект"]]
ref = ref.dropna(subset=["ид_объект"])
ref["ид_канала_данных"] = ref["ид_канала_данных"].astype(str)
obj_map = dict(zip(ref["ид_канала_данных"], ref["ид_объект"]))
rows = []
for task in ("fire", "access", "sensor", "wear"):
    d = pd.read_parquet(MODELS / ("tte_%s_holdout.parquet" % task))
    d["ид_канала_данных"] = d["ид_канала_данных"].astype(str)
    d["ид_объект"] = d["ид_канала_данных"].map(obj_map)
    d = d.dropna(subset=["ид_объект"])
    g = (d.assign(r300=d["risk30"], task=task,
              top=lambda x: (x["risk30"] > 0.5).astype(int))
         .groupby(["ид_объект", "task"])
         .agg(каналов=("ид_канала_данных", "count"),
              risk30_max=("r300", "max"),
              risk30_средн=("r300", "mean"),
              топ_каналы=("top", "sum"))
         .reset_index())
    rows.append(g)
l2 = pd.concat(rows, ignore_index=True)
l2 = l2.sort_values("risk30_max", ascending=False)
print("L2: объектов-строк:", len(l2))
print(l2.head(15).to_string(index=False))

# сохраняем пример для отчёта
l2.head(500).to_parquet(RESEARCH / "dataset" / "l2_object_risk.parquet", index=False)


L2: объектов-строк: 189
ид_объект   task  каналов  risk30_max  risk30_средн  топ_каналы
     5961 access      989         1.0      0.553449         544
     5132   wear      253         1.0      0.651783         169
     3902   wear     1086         1.0      0.905503        1075
     5657   wear     1589         1.0      0.885264        1573
     4369   wear      992         1.0      0.653428         673
      111   wear     1002         1.0      0.647857         630
     3388   wear     1817         1.0      0.593292        1110
     5675   wear     1553         1.0      0.886435        1508
     5218   wear      432         1.0      0.936673         426
     3360   wear     1007         1.0      0.836126         974
     5122   wear     1750         1.0      0.690740        1357
     5011   wear     1540         1.0      0.696030        1216
     5333   wear     1213         1.0      0.863943        1194
     4610   wear     1658         1.0      0.911841        1652
     5578   wear

In [5]:
# D. Отчёт соответствия ТЗ (вызов скрипта) + E. статус адаптеров
import importlib.util as iu
spec = iu.spec_from_file_location("rtrc", RESEARCH / "report_tz_compliance.py")
m = iu.module_from_spec(spec)
spec.loader.exec_module(m)
m.main()
print("TZ_COMPLIANCE_REPORT.md пересоздан")
print("E: адаптеры — см. docs/ADAPTERS_README.md (интерфейсы read-only, данные не поставлены)")


saved: D:\Downloads_D\lct_a8\research\docs\TZ_COMPLIANCE_REPORT.md
TZ_COMPLIANCE_REPORT.md пересоздан
E: адаптеры — см. docs/ADAPTERS_README.md (интерфейсы read-only, данные не поставлены)
